In [1]:
import os
import glob
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Dense, Dropout, GlobalAveragePooling1D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import StratifiedKFold
from scipy.signal import resample
import time
import json

import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
tf.get_logger().setLevel('ERROR')
import logging
logging.getLogger('tensorflow').setLevel(logging.ERROR)

print("TensorFlow version:", tf.__version__)


from google.colab import drive
drive.mount('/content/drive')

CHECKPOINT_DIR = '/content/drive/MyDrive/EOG_UserDependent_Checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"✓ Checkpoint directory: {CHECKPOINT_DIR}")


def save_fold_result(subj_id, fold_idx, result_dict):
 
    filename = f"{CHECKPOINT_DIR}/subj{subj_id}_fold{fold_idx}.json"
  
    serializable = {}
    for key, val in result_dict.items():
        if isinstance(val, np.ndarray):
            serializable[key] = val.tolist()
        else:
            serializable[key] = val
    with open(filename, 'w') as f:
        json.dump(serializable, f)
    print(f"  💾 Saved checkpoint: subj{subj_id}_fold{fold_idx}.json")

def load_fold_result(subj_id, fold_idx):
    
    filename = f"{CHECKPOINT_DIR}/subj{subj_id}_fold{fold_idx}.json"
    if os.path.exists(filename):
        with open(filename, 'r') as f:
            data = json.load(f)
    
        for key in ['cm1', 'cm2', 'cm3', 'cm_final']:
            if key in data:
                data[key] = np.array(data[key])
        return data
    return None

def is_fold_completed(subj_id, fold_idx):
  
    filename = f"{CHECKPOINT_DIR}/subj{subj_id}_fold{fold_idx}.json"
    return os.path.exists(filename)

# ============================================================
# 1. Download and Extract Original Dataset
# ============================================================

if not os.path.exists('./isolated'):
    !wget -O 'pone.0192684.s001.zip' 'https://doi.org/10.1371/journal.pone.0192684.s001'
    !unzip -o 'pone.0192684.s001.zip'
else:
    print("✓ Dataset already exists, skipping download.")

# ============================================================
# 2. Data Loading Function
# ============================================================
def load_isolated_data(base_dir='./isolated', target_length=1250):
    X_horiz_list, X_vert_list, X_diff_list = [], [], []
    y_list, subject_list = [], []

    for subj_id in range(1, 7):
        subj_dir = os.path.join(base_dir, f'{subj_id:03d}', 'isolated_strokes')
        if not os.path.exists(subj_dir):
            print(f"Warning: directory not found: {subj_dir}")
            continue

        csv_files = glob.glob(os.path.join(subj_dir, '*.csv'))

        for fpath in csv_files:
            fname = os.path.basename(fpath)
            parts = fname.replace('.csv', '').split('_')
            stroke_id = int(parts[3])

            if stroke_id < 1 or stroke_id > 12:
                continue

            df = pd.read_csv(fpath, header=None)
            if df.shape[1] < 2 or df.shape[0] < 10:
                continue

            v_signal = df.iloc[:, 0].values.astype(float)
            h_signal = df.iloc[:, 1].values.astype(float)

            h_resampled = resample(h_signal, target_length)
            v_resampled = resample(v_signal, target_length)
            d_resampled = h_resampled - v_resampled

            X_horiz_list.append(h_resampled)
            X_vert_list.append(v_resampled)
            X_diff_list.append(d_resampled)
            y_list.append(stroke_id - 1)
            subject_list.append(subj_id)

    X_horiz = np.array(X_horiz_list)
    X_vert = np.array(X_vert_list)
    X_diff = np.array(X_diff_list)
    y = np.array(y_list)
    subjects = np.array(subject_list)

    print(f"Total samples loaded: {len(y)}")
    for s in range(1, 7):
        print(f"  Subject {s}: {np.sum(subjects==s)} samples")

    return X_horiz, X_vert, X_diff, y, subjects

# Load data
X_horiz, X_vert, X_diff, y, subjects = load_isolated_data()

# Reshape
X_horiz = X_horiz.reshape(X_horiz.shape[0], X_horiz.shape[1], 1)
X_vert = X_vert.reshape(X_vert.shape[0], X_vert.shape[1], 1)
X_diff = X_diff.reshape(X_diff.shape[0], X_diff.shape[1], 1)
y_cat = to_categorical(y, num_classes=12)

# ============================================================
# 3. Model Architecture
# ============================================================
def create_1d_cnn(input_shape=(1250, 1)):
    model = Sequential([
        Conv1D(16, 3, activation='relu', input_shape=(1250, 1)), 
        MaxPooling1D(pool_size=2),
        Conv1D(32, 3, activation='relu'), 
        GlobalAveragePooling1D(), 
        Dense(32, activation='relu'),
        Dropout(0.5),
        Dense(12, activation='softmax')

    ])
    model.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# ============================================================
# 4. User-Dependent Evaluation Loop 
# ============================================================
unique_subjects = np.unique(subjects)
n_subjects = len(unique_subjects)


total_epochs = 450
batch_size = 16


def count_completed_folds(subj_id, max_folds=10):
    count = 0
    for f in range(max_folds):
        if is_fold_completed(subj_id, f):
            count += 1
        else:
            break
    return count

print(f"\n{'='*70}")
print(f"USER-DEPENDENT EVALUATION (with Resume capability)")
print(f"{'='*70}")

for subj_id in unique_subjects:
    print(f"\n{'─'*70}")
    print(f"Subject {subj_id}")
    print(f"{'─'*70}")

    # Extract data for this subject
    subj_mask = subjects == subj_id
    X_h_subj = X_horiz[subj_mask]
    X_v_subj = X_vert[subj_mask]
    X_d_subj = X_diff[subj_mask]
    y_subj = y[subj_mask]
    y_cat_subj = y_cat[subj_mask]

    n_samples = len(X_h_subj)
    n_unique_classes = len(np.unique(y_subj))
    print(f"  Samples: {n_samples}, Unique classes: {n_unique_classes}")

    # Adaptive k
    if n_samples < 50:
        k_folds = 5
    elif n_samples < 100:
        k_folds = 10
    else:
        k_folds = 10

    if n_unique_classes < k_folds:
        k_folds = max(2, n_unique_classes)

    # Stratified K-Fold
    skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)
    folds_list = list(skf.split(X_h_subj, y_subj))  

 
    completed_folds = count_completed_folds(subj_id, k_folds)
    if completed_folds > 0:
        print(f"  ✓ Found {completed_folds} previously completed folds, resuming from fold {completed_folds}")
    else:
        print(f"  Starting fresh, {k_folds} folds to process")

    for fold_idx, (train_idx, val_idx) in enumerate(folds_list):
       
        if is_fold_completed(subj_id, fold_idx):
            print(f"  Fold {fold_idx+1}/{k_folds}: ✓ Already completed (loaded from checkpoint)")
            continue

        print(f"  Fold {fold_idx+1}/{k_folds}: Training...")

        X_h_train, X_h_val = X_h_subj[train_idx], X_h_subj[val_idx]
        X_v_train, X_v_val = X_v_subj[train_idx], X_v_subj[val_idx]
        X_d_train, X_d_val = X_d_subj[train_idx], X_d_subj[val_idx]
        y_train_cat, y_val_cat = y_cat_subj[train_idx], y_cat_subj[val_idx]
        y_val_idx = np.argmax(y_val_cat, axis=1)

        # Create fresh models
        part1_model = create_1d_cnn()
        part2_model = create_1d_cnn()
        part3_model = create_1d_cnn()

        # Train branches
        part1_model.fit(X_h_train, y_train_cat, epochs=total_epochs,
                        batch_size=batch_size, verbose=0, shuffle=True)
        part2_model.fit(X_v_train, y_train_cat, epochs=total_epochs,
                        batch_size=batch_size, verbose=0, shuffle=True)
        part3_model.fit(X_d_train, y_train_cat, epochs=total_epochs,
                        batch_size=batch_size, verbose=0, shuffle=True)

        # Freeze branches
        part1_model.trainable = False
        part2_model.trainable = False
        part3_model.trainable = False

        # Build fusion
        p1_train = part1_model.predict(X_h_train, verbose=0)
        p2_train = part2_model.predict(X_v_train, verbose=0)
        p3_train = part3_model.predict(X_d_train, verbose=0)
        combined_train = np.concatenate([p1_train, p2_train, p3_train], axis=1)

        fusion_model = Sequential([
            Dense(16, activation='relu', input_shape=(36,)),
            Dense(12, activation='softmax')
        ])
        fusion_model.compile(optimizer='adam', loss='categorical_crossentropy',
                             metrics=['accuracy'])
        fusion_model.fit(combined_train, y_train_cat, epochs=total_epochs,
                         batch_size=batch_size, verbose=0, shuffle=True)

        # Evaluation
        p1_pred = np.argmax(part1_model.predict(X_h_val, verbose=0), axis=1)
        p2_pred = np.argmax(part2_model.predict(X_v_val, verbose=0), axis=1)
        p3_pred = np.argmax(part3_model.predict(X_d_val, verbose=0), axis=1)

        p1_test = part1_model.predict(X_h_val, verbose=0)
        p2_test = part2_model.predict(X_v_val, verbose=0)
        p3_test = part3_model.predict(X_d_val, verbose=0)
        combined_test = np.concatenate([p1_test, p2_test, p3_test], axis=1)
        final_pred = np.argmax(fusion_model.predict(combined_test, verbose=0), axis=1)

        # Confusion matrices
        cm1 = confusion_matrix(y_val_idx, p1_pred, labels=range(12))
        cm2 = confusion_matrix(y_val_idx, p2_pred, labels=range(12))
        cm3 = confusion_matrix(y_val_idx, p3_pred, labels=range(12))
        cm_final = confusion_matrix(y_val_idx, final_pred, labels=range(12))

        # accuracy 
        acc1 = np.trace(cm1) / np.sum(cm1) if np.sum(cm1) > 0 else 0
        acc2 = np.trace(cm2) / np.sum(cm2) if np.sum(cm2) > 0 else 0
        acc3 = np.trace(cm3) / np.sum(cm3) if np.sum(cm3) > 0 else 0
        acc_final = np.trace(cm_final) / np.sum(cm_final) if np.sum(cm_final) > 0 else 0

        print(f"    Accuracies: P1={acc1:.4f}, P2={acc2:.4f}, P3={acc3:.4f}, Final={acc_final:.4f}")

        # (Checkpoint)
        fold_result = {
            'cm1': cm1,
            'cm2': cm2,
            'cm3': cm3,
            'cm_final': cm_final,
            'n_samples_train': len(train_idx),
            'n_samples_val': len(val_idx),
            'k_folds': k_folds,
            'n_unique_classes': int(n_unique_classes)
        }
        save_fold_result(subj_id, fold_idx, fold_result)

# ============================================================
# 5. Aggregate All Results 
# ============================================================
print(f"\n{'='*70}")
print("AGGREGATING ALL RESULTS FROM CHECKPOINTS")
print(f"{'='*70}")

ud_results = []
all_user_metrics = {}

for subj_id in unique_subjects:
    subj_mask = subjects == subj_id
    n_samples = np.sum(subj_mask)
    n_unique_classes = len(np.unique(y[subj_mask]))

   
    subj_folds_cm = {'part1': [], 'part2': [], 'part3': [], 'final': []}
    k_folds = None

    for fold_idx in range(20):  
        result = load_fold_result(subj_id, fold_idx)
        if result is None:
            break
        subj_folds_cm['part1'].append(result['cm1'])
        subj_folds_cm['part2'].append(result['cm2'])
        subj_folds_cm['part3'].append(result['cm3'])
        subj_folds_cm['final'].append(result['cm_final'])
        k_folds = result['k_folds']

    if k_folds is None:
        print(f"⚠ Warning: No checkpoints found for subject {subj_id}")
        continue

    # Aggregate across folds
    agg_cm_part1 = np.sum(subj_folds_cm['part1'], axis=0)
    agg_cm_part2 = np.sum(subj_folds_cm['part2'], axis=0)
    agg_cm_part3 = np.sum(subj_folds_cm['part3'], axis=0)
    agg_cm_final = np.sum(subj_folds_cm['final'], axis=0)

    acc_part1 = np.trace(agg_cm_part1) / np.sum(agg_cm_part1)
    acc_part2 = np.trace(agg_cm_part2) / np.sum(agg_cm_part2)
    acc_part3 = np.trace(agg_cm_part3) / np.sum(agg_cm_part3)
    acc_final = np.trace(agg_cm_final) / np.sum(agg_cm_final)

    print(f"Subject {subj_id}: Part1={acc_part1:.4f}, Part2={acc_part2:.4f}, "
          f"Part3={acc_part3:.4f}, Final={acc_final:.4f}")

    ud_results.append({
        'subject': subj_id,
        'n_samples': int(n_samples),
        'n_classes': int(n_unique_classes),
        'k_folds': k_folds,
        'acc_part1': acc_part1,
        'acc_part2': acc_part2,
        'acc_part3': acc_part3,
        'acc_final': acc_final
    })

    all_user_metrics[subj_id] = {
        'agg_part1': agg_cm_part1,
        'agg_part2': agg_cm_part2,
        'agg_part3': agg_cm_part3,
        'agg_final': agg_cm_final
    }

# ============================================================
# 6. Final Aggregate Across Users
# ============================================================
df_ud = pd.DataFrame(ud_results)

print(f"\n{'='*70}")
print("AGGREGATED USER-DEPENDENT RESULTS")
print(f"{'='*70}")
print(df_ud.to_string(index=False))

# Weighted average
total_samples = df_ud['n_samples'].sum()
w_acc_p1 = np.sum(df_ud['acc_part1'] * df_ud['n_samples']) / total_samples
w_acc_p2 = np.sum(df_ud['acc_part2'] * df_ud['n_samples']) / total_samples
w_acc_p3 = np.sum(df_ud['acc_part3'] * df_ud['n_samples']) / total_samples
w_acc_final = np.sum(df_ud['acc_final'] * df_ud['n_samples']) / total_samples

# Simple mean
m_acc_p1 = df_ud['acc_part1'].mean()
m_acc_p2 = df_ud['acc_part2'].mean()
m_acc_p3 = df_ud['acc_part3'].mean()
m_acc_final = df_ud['acc_final'].mean()

print(f"\n{'Branch':<15} {'Weighted Acc':<15} {'Mean Acc':<15} {'Std Dev':<15}")
print("-" * 60)
print(f"{'Part1 (Horiz)':<15} {w_acc_p1:<15.4f} {m_acc_p1:<15.4f} {df_ud['acc_part1'].std():<15.4f}")
print(f"{'Part2 (Vert)':<15} {w_acc_p2:<15.4f} {m_acc_p2:<15.4f} {df_ud['acc_part2'].std():<15.4f}")
print(f"{'Part3 (Diff)':<15} {w_acc_p3:<15.4f} {m_acc_p3:<15.4f} {df_ud['acc_part3'].std():<15.4f}")
print(f"{'FUSED (Final)':<15} {w_acc_final:<15.4f} {m_acc_final:<15.4f} {df_ud['acc_final'].std():<15.4f}")

# Global aggregated confusion matrix
global_cm_part1 = sum(all_user_metrics[s]['agg_part1'] for s in all_user_metrics.keys())
global_cm_part2 = sum(all_user_metrics[s]['agg_part2'] for s in all_user_metrics.keys())
global_cm_part3 = sum(all_user_metrics[s]['agg_part3'] for s in all_user_metrics.keys())
global_cm_final = sum(all_user_metrics[s]['agg_final'] for s in all_user_metrics.keys())

print(f"\nOverall User-Dependent Accuracy (global aggregated CM):")
print(f"  Part1: {np.trace(global_cm_part1)/np.sum(global_cm_part1):.4f}")
print(f"  Part2: {np.trace(global_cm_part2)/np.sum(global_cm_part2):.4f}")
print(f"  Part3: {np.trace(global_cm_part3)/np.sum(global_cm_part3):.4f}")
print(f"  Final: {np.trace(global_cm_final)/np.sum(global_cm_final):.4f}")

# ============================================================
# 7. save results
# ============================================================
df_ud.to_csv(f"{CHECKPOINT_DIR}/final_results.csv", index=False)
np.save(f"{CHECKPOINT_DIR}/global_cm_final.npy", global_cm_final)

summary = {
    'Protocol': 'User-Dependent',
    'Weighted_Acc_Final': float(w_acc_final),
    'Mean_Acc_Final': float(m_acc_final),
    'Std_Acc_Final': float(df_ud['acc_final'].std()),
    'Overall_Accuracy': float(np.trace(global_cm_final)/np.sum(global_cm_final))
}

with open(f"{CHECKPOINT_DIR}/final_summary.json", 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\n✓ All final results saved to {CHECKPOINT_DIR}")
print("✓ If Colab disconnects, simply re-run this notebook to resume!")

TensorFlow version: 2.20.0
Mounted at /content/drive
✓ Checkpoint directory: /content/drive/MyDrive/EOG_UserDependent_Checkpoints
--2026-09-11 17:13:26--  https://doi.org/10.1371/journal.pone.0192684.s001
Resolving doi.org (doi.org)... 104.26.4.132, 172.67.69.3, 104.26.5.132, ...
Connecting to doi.org (doi.org)|104.26.4.132|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://dx.plos.org/10.1371/journal.pone.0192684.s001 [following]
--2026-09-11 17:13:26--  https://dx.plos.org/10.1371/journal.pone.0192684.s001
Resolving dx.plos.org (dx.plos.org)... 35.190.43.188
Connecting to dx.plos.org (dx.plos.org)|35.190.43.188|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://journals.plos.org/plosone/doi?id=10.1371/journal.pone.0192684.s001 [following]
--2026-09-11 17:13:27--  https://journals.plos.org/plosone/doi?id=10.1371/journal.pone.0192684.s001
Resolving journals.plos.org (journals.plos.org)... 35.190.43.188


In [2]:
# ============================================================
# 1. Mean Confusion Matrices (User-Dependent)
# ============================================================
mean_cm_part1_ud = np.mean([all_user_metrics[s]['agg_part1'] for s in unique_subjects], axis=0)
mean_cm_part2_ud = np.mean([all_user_metrics[s]['agg_part2'] for s in unique_subjects], axis=0)
mean_cm_part3_ud = np.mean([all_user_metrics[s]['agg_part3'] for s in unique_subjects], axis=0)
mean_cm_final_ud = np.mean([all_user_metrics[s]['agg_final'] for s in unique_subjects], axis=0)

print("\n" + "="*70)
print("MEAN CONFUSION MATRICES (User-Dependent, averaged across users)")
print("="*70)
print("\nMean CM - Part1:\n", mean_cm_part1_ud)
print("\nMean CM - Part2:\n", mean_cm_part2_ud)
print("\nMean CM - Part3:\n", mean_cm_part3_ud)
print("\nMean CM - Final:\n", mean_cm_final_ud)

# Save
np.save(f'{CHECKPOINT_DIR}/ud_mean_cm_part1.npy', mean_cm_part1_ud)
np.save(f'{CHECKPOINT_DIR}/ud_mean_cm_part2.npy', mean_cm_part2_ud)
np.save(f'{CHECKPOINT_DIR}/ud_mean_cm_part3.npy', mean_cm_part3_ud)
np.save(f'{CHECKPOINT_DIR}/ud_mean_cm_final.npy', mean_cm_final_ud)
np.save(f'{CHECKPOINT_DIR}/ud_global_cm_final.npy', global_cm_final)
print(f"\n✓ Confusion matrices saved to {CHECKPOINT_DIR}")

# ============================================================
# 2. Metrics Calculation Function
# ============================================================
def calculate_metrics_ud(cm, model_name):
    num_classes = cm.shape[0]
    precision, recall, f1 = [], [], []

    for i in range(num_classes):
        tp = cm[i, i]
        fp = np.sum(cm[:, i]) - tp
        fn = np.sum(cm[i, :]) - tp

        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1_scr = 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0

        precision.append(prec)
        recall.append(rec)
        f1.append(f1_scr)

    macro_p = np.mean(precision)
    macro_r = np.mean(recall)
    macro_f1 = np.mean(f1)
    accuracy = np.trace(cm) / np.sum(cm)

    print(f"\n--- {model_name} (User-Dependent) ---")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Macro Precision: {macro_p:.4f}")
    print(f"Macro Recall: {macro_r:.4f}")
    print(f"Macro F1: {macro_f1:.4f}")

    return accuracy, macro_p, macro_r, macro_f1

# ============================================================
# 3. Print All Metrics
# ============================================================
print("\n" + "="*70)
print("METRICS PER BRANCH (User-Dependent)")
print("="*70)

acc_p1, prec_p1, rec_p1, f1_p1 = calculate_metrics_ud(global_cm_part1, "Part1 (Horizontal)")
acc_p2, prec_p2, rec_p2, f1_p2 = calculate_metrics_ud(global_cm_part2, "Part2 (Vertical)")
acc_p3, prec_p3, rec_p3, f1_p3 = calculate_metrics_ud(global_cm_part3, "Part3 (Difference)")
acc_fin, prec_fin, rec_fin, f1_fin = calculate_metrics_ud(global_cm_final, "Final (Fused)")

# ============================================================
# 4. Per-User Accuracy Breakdown
# ============================================================
print("\n" + "="*70)
print("PER-USER ACCURACY BREAKDOWN")
print("="*70)
print(f"{'Subject':<10} {'Samples':<10} {'Classes':<10} {'Part1':<12} {'Part2':<12} {'Part3':<12} {'Final':<12}")
print("-" * 78)
for _, row in df_ud.iterrows():
    print(f"{row['subject']:<10} {row['n_samples']:<10} {row['n_classes']:<10} "
          f"{row['acc_part1']:<12.4f} {row['acc_part2']:<12.4f} "
          f"{row['acc_part3']:<12.4f} {row['acc_final']:<12.4f}")

# ============================================================
# 5. Mean & Variance Across Users
# ============================================================
print("\n" + "="*70)
print("MEAN & VARIANCE ACROSS USERS")
print("="*70)
print(f"Part1 - Mean: {df_ud['acc_part1'].mean():.4f}, "
      f"Variance: {df_ud['acc_part1'].var():.4f}, "
      f"Std: {df_ud['acc_part1'].std():.4f}")
print(f"Part2 - Mean: {df_ud['acc_part2'].mean():.4f}, "
      f"Variance: {df_ud['acc_part2'].var():.4f}, "
      f"Std: {df_ud['acc_part2'].std():.4f}")
print(f"Part3 - Mean: {df_ud['acc_part3'].mean():.4f}, "
      f"Variance: {df_ud['acc_part3'].var():.4f}, "
      f"Std: {df_ud['acc_part3'].std():.4f}")
print(f"Final - Mean: {df_ud['acc_final'].mean():.4f}, "
      f"Variance: {df_ud['acc_final'].var():.4f}, "
      f"Std: {df_ud['acc_final'].std():.4f}")


# ============================================================
# 6. Final Summary Table
# ============================================================
print("\n" + "="*70)
print("FINAL SUMMARY - USER-DEPENDENT EVALUATION")
print("="*70)
print(f"{'Branch':<15} {'Accuracy':<12} {'Macro P':<12} {'Macro R':<12} {'Macro F1':<12}")
print("-" * 63)
print(f"{'Part1 (Horiz)':<15} {acc_p1:<12.4f} {prec_p1:<12.4f} {rec_p1:<12.4f} {f1_p1:<12.4f}")
print(f"{'Part2 (Vert)':<15} {acc_p2:<12.4f} {prec_p2:<12.4f} {rec_p2:<12.4f} {f1_p2:<12.4f}")
print(f"{'Part3 (Diff)':<15} {acc_p3:<12.4f} {prec_p3:<12.4f} {rec_p3:<12.4f} {f1_p3:<12.4f}")
print(f"{'FUSED':<15} {acc_fin:<12.4f} {prec_fin:<12.4f} {rec_fin:<12.4f} {f1_fin:<12.4f}")

# ============================================================
# 7. Save Summary
# ============================================================
summary = {
    'Protocol': 'User-Dependent (per-subject model)',
    'Part1_Accuracy': float(acc_p1), 'Part1_Precision': float(prec_p1),
    'Part1_Recall': float(rec_p1), 'Part1_F1': float(f1_p1),
    'Part2_Accuracy': float(acc_p2), 'Part2_Precision': float(prec_p2),
    'Part2_Recall': float(rec_p2), 'Part2_F1': float(f1_p2),
    'Part3_Accuracy': float(acc_p3), 'Part3_Precision': float(prec_p3),
    'Part3_Recall': float(rec_p3), 'Part3_F1': float(f1_p3),
    'Final_Accuracy': float(acc_fin), 'Final_Precision': float(prec_fin),
    'Final_Recall': float(rec_fin), 'Final_F1': float(f1_fin),
}

with open(f'{CHECKPOINT_DIR}/user_dependent_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\n✓ Results saved to '{CHECKPOINT_DIR}/user_dependent_summary.json'")
print("✓ Per-subject results saved in df_ud DataFrame")


# ============================================================
# 8. Per-Subject Performance Visualization Data
# ============================================================
print("\n" + "="*70)
print("USER-DEPENDENT CONSISTENCY ANALYSIS")
print("="*70)

best_user = df_ud.loc[df_ud['acc_final'].idxmax()]
worst_user = df_ud.loc[df_ud['acc_final'].idxmin()]

print(f"Best performing subject:  #{int(best_user['subject'])} "
      f"with {best_user['acc_final']:.4f} accuracy")
print(f"Worst performing subject: #{int(worst_user['subject'])} "
      f"with {worst_user['acc_final']:.4f} accuracy")
print(f"Performance gap: {best_user['acc_final'] - worst_user['acc_final']:.4f} "
      f"({(best_user['acc_final'] - worst_user['acc_final'])*100:.2f} pp)")
print(f"Inter-subject std dev: {df_ud['acc_final'].std():.4f}")


MEAN CONFUSION MATRICES (User-Dependent, averaged across users)

Mean CM - Part1:
 [[1.83333333 0.         0.16666667 1.16666667 1.5        0.33333333
  0.         1.16666667 2.33333333 1.5        0.         0.        ]
 [0.         4.16666667 0.16666667 0.33333333 0.         1.5
  0.         0.         2.         0.         0.         1.83333333]
 [0.16666667 0.16666667 3.         0.16666667 0.16666667 0.33333333
  1.83333333 0.         1.         0.         3.16666667 0.        ]
 [1.5        0.         0.16666667 3.16666667 0.66666667 1.
  0.         1.         1.         1.16666667 0.16666667 0.16666667]
 [1.16666667 0.33333333 0.         1.66666667 2.83333333 1.
  0.         0.83333333 1.5        0.66666667 0.         0.16666667]
 [0.         1.         0.33333333 0.5        0.66666667 5.33333333
  0.         0.33333333 1.         0.         0.5        0.5       ]
 [0.         0.         1.16666667 0.33333333 0.         0.33333333
  5.16666667 0.         0.5        0.         2.6